# Tests for src/

Regression tests for the functions extracted into src/, checking their
output against known-good values already verified in
00_building_the_model.ipynb, plus a few edge cases (a player with no
prior season, a player with no current-season games, and the formal
correctness properties of the comparison model).

Run this any time src/ changes, before trusting it in a weekly
prediction notebook.

In [1]:
# Always clear any previously downloaded copies first, wget silently
# saves as .1, .2, etc. rather than overwriting, which would otherwise
# leave an old, stale version of these files in place even after
# re-running this cell.
!rm -f data_pull.py data_pull.py.* scoring.py scoring.py.* stats.py stats.py.* matchup.py matchup.py.*

!wget -q -O data_pull.py https://raw.githubusercontent.com/avieno33/fantasy-hockey-weekly-predictions/main/src/data_pull.py
!wget -q -O scoring.py https://raw.githubusercontent.com/avieno33/fantasy-hockey-weekly-predictions/main/src/scoring.py
!wget -q -O stats.py https://raw.githubusercontent.com/avieno33/fantasy-hockey-weekly-predictions/main/src/stats.py
!wget -q -O matchup.py https://raw.githubusercontent.com/avieno33/fantasy-hockey-weekly-predictions/main/src/matchup.py

from data_pull import get_player_id
from data_pull import get_game_log, get_boxscore
from stats import clean_game_log, ALL_SKATER_RAW_FIELDS
from scoring import SKATER_FIELD_MAP, GOALIE_FIELD_MAP
from stats import get_player_estimates
from matchup import compare_players

# same default league scoring used in the base notebook, needed here
# only to reproduce known-good results, not tied to any real league
skater_scoring = {
    "G": 2, "A": 1, "PIM": -0.5, "PPG": 0.5, "SHG": 2,
    "GWG": 1, "SOG": 0.1, "HIT": 0.25, "BLK": 0.25,
}
goalie_scoring = {"W": 5, "GA": -1, "SV": 0.1, "SHO": 5}

LAST_SEASON = "20252026"
PRIOR_SEASON = "20242025"
CURRENT_SEASON = "20262027"

checks_passed = 0
checks_total = 0

def check(condition, description):
    global checks_passed, checks_total
    checks_total += 1
    if condition:
        checks_passed += 1
        print(f"[pass] {description}")
    else:
        print(f"[FAIL] {description}")

def close(a, b, tolerance=0.05):
    return abs(a - b) < tolerance

## Test 1, a known skater (McDavid), regression against verified values

In [2]:
mcdavid_id = get_player_id("Connor McDavid")
print(type(mcdavid_id), mcdavid_id)

<class 'int'> 8478402


In [3]:
mcdavid_id = get_player_id("Connor McDavid")
mcdavid = get_player_estimates(mcdavid_id, "C", skater_scoring, SKATER_FIELD_MAP,
                                 season=LAST_SEASON, prior_season=PRIOR_SEASON)

check(mcdavid is not None, "McDavid estimates returned, not None")
check(mcdavid["games_played"] == 82, f"McDavid games played is 82 (got {mcdavid['games_played']})")
check(close(mcdavid["season_mu"], 2.69), f"McDavid season_mu close to 2.69 (got {mcdavid['season_mu']:.2f})")
check(close(mcdavid["season_sigma"], 2.26), f"McDavid season_sigma close to 2.26 (got {mcdavid['season_sigma']:.2f})")
check(close(mcdavid["recent_mu"], 2.79), f"McDavid recent_mu close to 2.79 (got {mcdavid['recent_mu']:.2f})")

Loading game log for 8478402, 20252026 from cache, 0.2 hours old
Loading game log for 8478402, 20242025 from cache, 0.0 hours old
[pass] McDavid estimates returned, not None
[pass] McDavid games played is 82 (got 82)
[pass] McDavid season_mu close to 2.69 (got 2.68)
[pass] McDavid season_sigma close to 2.26 (got 2.26)
[pass] McDavid recent_mu close to 2.79 (got 2.79)


## Test 2, a known defenseman (Weegar), regression against verified values

In [4]:
weegar_id = get_player_id("MacKenzie Weegar")
weegar = get_player_estimates(weegar_id, "D", skater_scoring, SKATER_FIELD_MAP,
                                season=LAST_SEASON, prior_season=PRIOR_SEASON)

check(weegar is not None, "Weegar estimates returned, not None")
check(close(weegar["season_mu"], 1.24), f"Weegar season_mu close to 1.24 (got {weegar['season_mu']:.2f})")
check(close(weegar["season_sigma"], 1.64), f"Weegar season_sigma close to 1.64 (got {weegar['season_sigma']:.2f})")

Pulling fresh game log for 8477346, 20252026
Pulling fresh game log for 8477346, 20242025
[pass] Weegar estimates returned, not None
[pass] Weegar season_mu close to 1.24 (got 1.22)
[pass] Weegar season_sigma close to 1.64 (got 1.61)


## Test 3, edge case, a goalie with one game and no prior season (Brossoit)

This is the case that originally exposed the clean_goalie_log bug,
worth keeping as a permanent regression test given how much trouble
it caused.

In [5]:
brossoit_id = get_player_id("Laurent Brossoit")
brossoit = get_player_estimates(brossoit_id, "G", goalie_scoring, GOALIE_FIELD_MAP,
                                  season=LAST_SEASON, prior_season=PRIOR_SEASON)

check(brossoit is not None, "Brossoit estimates returned, not None, despite sparse data")
check(brossoit["games_played"] == 1, f"Brossoit games played is 1 (got {brossoit['games_played']})")
check(close(brossoit["season_mu"], -4.3), f"Brossoit season_mu close to -4.3 (got {brossoit['season_mu']:.2f})")
check(brossoit["season_sigma"] == 0, "Brossoit season_sigma is 0 with a single game, no prior to blend against")

Pulling fresh game log for 8476316, 20252026
Pulling fresh game log for 8476316, 20242025
No games found for 8476316 in 20242025, player likely wasn't in the NHL that season
[pass] Brossoit estimates returned, not None, despite sparse data
[pass] Brossoit games played is 1 (got 1)
[pass] Brossoit season_mu close to -4.3 (got -4.30)
[pass] Brossoit season_sigma is 0 with a single game, no prior to blend against


## Test 4, edge case, a player with zero games in a season that hasn't started

get_player_estimates should return None cleanly, not raise an error.

In [6]:
result = get_player_estimates(mcdavid_id, "C", skater_scoring, SKATER_FIELD_MAP,
                                season=CURRENT_SEASON, prior_season=PRIOR_SEASON)

check(result is None, "Empty current season returns None instead of crashing")

Pulling fresh game log for 8478402, 20262027
No games found for 8478402 in 20262027, season may not have started yet
[pass] Empty current season returns None instead of crashing


## Test 5, compare_players, formal correctness properties

Independent of any specific player, these must hold for any valid
input, not just cases that happen to work.

In [7]:
a = {"season_mu": 3.0, "season_sigma": 1.5}
b = {"season_mu": 2.0, "season_sigma": 1.0}
result_ab = compare_players(a, b)
result_ba = compare_players(b, a)
check(close(result_ab["probability_a_outperforms_b"] + result_ba["probability_a_outperforms_b"], 1.0, 1e-9),
      "P(A>B) and P(B>A) sum to 1")

same = {"season_mu": 2.5, "season_sigma": 1.2}
result_same = compare_players(same, dict(same))
check(close(result_same["probability_a_outperforms_b"], 0.5, 1e-9), "identical players give exactly 50%")

zero_sigma_a = {"season_mu": 3.0, "season_sigma": 0.0}
zero_sigma_b = {"season_mu": 2.0, "season_sigma": 0.0}
result_zero = compare_players(zero_sigma_a, zero_sigma_b)
check(result_zero["probability_a_outperforms_b"] == 1.0, "zero volatility resolves deterministically, no crash")

# a real assert too, as a hard stop, this one should never be false
# under any circumstance, worth failing loudly rather than just logging
assert 0.0 <= result_ab["probability_a_outperforms_b"] <= 1.0, "probability must be a valid value between 0 and 1"
print("[pass] probability is within valid bounds (0 to 1), enforced with assert")

[pass] P(A>B) and P(B>A) sum to 1
[pass] identical players give exactly 50%
[pass] zero volatility resolves deterministically, no crash
[pass] probability is within valid bounds (0 to 1), enforced with assert


## Test 6, compare_players, regression against a known real result

McDavid vs. Weegar previously came out to 69.9%, worth confirming the
extracted version reproduces this exactly, not just structurally
correct in isolation.

In [8]:
result = compare_players(mcdavid, weegar, "McDavid", "Weegar")
check(close(result["probability_a_outperforms_b"], 0.699, 0.02),
      f"McDavid vs Weegar close to 69.9% (got {result['probability_a_outperforms_b']:.1%})")

[pass] McDavid vs Weegar close to 69.9% (got 70.0%)


## Summary

In [9]:
print(f"\n{checks_passed}/{checks_total} checks passed")
if checks_passed < checks_total:
    print("Do not trust src/ in week01 until every check above passes.")
else:
    print("All checks passed, src/ is verified against known-good results.")


17/17 checks passed
All checks passed, src/ is verified against known-good results.
